In [2]:
import sqlite3 
import pandas as pd 

conn= sqlite3.connect('flightdelay_project.db')

### BTS Flight Dataset
This dataset contains U.S. domestic flight records from January to April 2025, including routes, schedules, distances, and delay status.

Source: U.S. Bureau of Transportation Statistics (BTS)
- Loading dataset into SQLite database: 

In [3]:
#load flight data into SQL database
flights_df= pd.read_csv(
    r"C:\Users\what1\OneDrive\Documents\Flight Delay Prediction Model\data\raw\flightdata\flights_2025_jan_apr.csv"
)
flights_df.to_sql(
    "flights_raw",
    conn, if_exists="replace", index=False
)

2229453

### Flight Data Structure Check
This section checks the row count, date & month range, and required columns in the raw BTS flight data for month range January to April of 2025.

In [4]:
q1_rowcount= """ SELECT COUNT(*) AS row_count FROM flights_raw; """ 
pd.read_sql_query(q1_rowcount, conn)

,row_count
0,2229453


In [5]:
q2_table_info= """
PRAGMA table_info (flights_raw);
"""

pd.read_sql_query(q2_table_info, conn)

,cid,name,type,notnull,dflt_value,pk
0,0,MONTH,INTEGER,0,None,0
1,1,DAY_OF_MONTH,INTEGER,0,None,0
2,2,DAY_OF_WEEK,INTEGER,0,None,0
3,3,FL_DATE,TEXT,0,None,0
4,4,OP_UNIQUE_CARRIER,TEXT,0,None,0
5,5,TAIL_NUM,TEXT,0,None,0
6,6,ORIGIN_AIRPORT_ID,INTEGER,0,None,0
7,7,ORIGIN,TEXT,0,None,0
8,8,ORIGIN_CITY_NAME,TEXT,0,None,0
9,9,DEST_AIRPORT_ID,INTEGER,0,None,0


In [6]:
#check date ranges 
q3_daterange= """ 
SELECT MIN (FL_DATE) AS start_date, 
MAX (FL_DATE) AS end_date 
FROM flights_raw;
"""

pd.read_sql_query(q3_daterange, conn)

,start_date,end_date
0,1/1/2025 12:00:00 AM,4/9/2025 12:00:00 AM


In [7]:

q4_rowcount_by_month = """
SELECT MONTH, COUNT(*) AS ROW_COUNT, 
MIN (DAY_OF_MONTH) AS FIRST_DAY,
MAX (DAY_OF_MONTH) AS LAST_DAY
FROM flights_raw
GROUP BY MONTH
ORDER BY MONTH;"""


pd.read_sql_query(q4_rowcount_by_month, conn)

,MONTH,ROW_COUNT,FIRST_DAY,LAST_DAY
0,1,539747,1,31
1,2,504884,1,28
2,3,600872,1,31
3,4,583950,1,30


In [8]:
q5_columncheck="""
SELECT 
 MONTH, 
 DAY_OF_MONTH,
 DAY_OF_WEEK, 
 FL_DATE, 
 OP_UNIQUE_CARRIER, 
 ORIGIN, 
 DEST,
 CRS_DEP_TIME,
 DISTANCE, 
 ARR_DEL15, 
 CANCELLED, 
 DIVERTED
FROM flights_raw
LIMIT 10;
"""

pd.read_sql_query(q5_columncheck, conn)

,MONTH,DAY_OF_MONTH,DAY_OF_WEEK,FL_DATE,OP_UNIQUE_CARRIER,ORIGIN,DEST,CRS_DEP_TIME,DISTANCE,ARR_DEL15,CANCELLED,DIVERTED
0,1,1,3,1/1/2025 12:00:00 AM,AA,JFK,SFO,600,2586.0,0.0,0.0,0.0
1,1,1,3,1/1/2025 12:00:00 AM,AA,SFO,JFK,1030,2586.0,0.0,0.0,0.0
2,1,1,3,1/1/2025 12:00:00 AM,AA,SAT,CLT,819,1095.0,0.0,0.0,0.0
3,1,1,3,1/1/2025 12:00:00 AM,AA,BOS,LAX,801,2611.0,0.0,0.0,0.0
4,1,1,3,1/1/2025 12:00:00 AM,AA,JFK,LAX,2100,2475.0,0.0,0.0,0.0
5,1,1,3,1/1/2025 12:00:00 AM,AA,LAX,JFK,1130,2475.0,0.0,0.0,0.0
6,1,1,3,1/1/2025 12:00:00 AM,AA,CMH,PHX,746,1670.0,0.0,0.0,0.0
7,1,1,3,1/1/2025 12:00:00 AM,AA,PHX,OMA,2025,1037.0,0.0,0.0,0.0
8,1,1,3,1/1/2025 12:00:00 AM,AA,PHX,SMF,1104,647.0,1.0,0.0,0.0
9,1,1,3,1/1/2025 12:00:00 AM,AA,SMF,PHX,1259,647.0,0.0,0.0,0.0


In [9]:
q6_cancelled_diverted= """
SELECT CANCELLED, DIVERTED, COUNT(*) AS ROW_COUNT FROM flights_raw
GROUP BY CANCELLED, DIVERTED;
"""
pd.read_sql_query(q6_cancelled_diverted, conn)

,CANCELLED,DIVERTED,ROW_COUNT
0,0.0,0.0,2188776
1,0.0,1.0,5123
2,1.0,0.0,35554


In [10]:
q7_delay_labels= """
SELECT ARR_DEL15, COUNT(*) AS ROW_COUNT 
FROM FLIGHTS_RAW
WHERE ARR_DEL15 IS NOT NULL
GROUP BY ARR_DEL15;"""
pd.read_sql_query(q7_delay_labels, conn)

,ARR_DEL15,ROW_COUNT
0,0.0,1757906
1,1.0,430870


In [11]:
q8_missing_blank="""
SELECT 
SUM (CASE WHEN MONTH IS NULL THEN 1 ELSE 0 END) AS month_missing,
SUM (CASE WHEN DAY_OF_MONTH IS NULL THEN 1 ELSE 0 END) AS dayofmonth_missing,
SUM (CASE WHEN DAY_OF_WEEK IS NULL THEN 1 ELSE 0 END) AS dayofweek_missing,
SUM (CASE WHEN OP_UNIQUE_CARRIER IS NULL THEN 1 ELSE 0 END) AS carrier_missing,
SUM (CASE WHEN TRIM(OP_UNIQUE_CARRIER)= '' THEN 1 ELSE 0 END) AS carrier_blank,
SUM (CASE WHEN ORIGIN IS NULL THEN 1 ELSE 0 END) AS origin_missing, 
SUM (CASE WHEN TRIM(ORIGIN) = '' THEN 1 ELSE 0 END) AS origin_blank, 
SUM (CASE WHEN DEST IS NULL THEN 1 ELSE 0 END) AS dest_missing,
sum (CASE WHEN TRIM(DEST)='' THEN 1 ELSE 0 END) AS dest_blank, 
SUM (CASE WHEN CRS_DEP_TIME IS NULL THEN 1 ELSE 0 END) AS deptime_missing,
SUM (CASE WHEN DISTANCE IS NULL THEN 1 ELSE 0 END) AS distance_missing, 
SUM (CASE WHEN ARR_DEL15 IS NULL THEN 1 ELSE 0 END) AS arrdel15_missing
FROM flights_raw 
WHERE DIVERTED=0
AND CANCELLED =0;"""

pd.read_sql_query(q8_missing_blank, conn)

,month_missing,dayofmonth_missing,dayofweek_missing,carrier_missing,carrier_blank,origin_missing,origin_blank,dest_missing,dest_blank,deptime_missing,distance_missing,arrdel15_missing
0,0,0,0,0,0,0,0,0,0,0,0,0


In [12]:
q9_invalids= """
SELECT 
 SUM (CASE WHEN MONTH NOT BETWEEN 1 AND 12 THEN 1 ELSE 0 END) AS invalid_month,
 SUM (CASE WHEN DAY_OF_WEEK NOT BETWEEN 1 AND 7 THEN 1 ELSE 0 END) AS invalid_week,
 SUM (CASE WHEN DAY_OF_MONTH NOT BETWEEN 1 AND 31 THEN 1 ELSE 0 END) AS invalid_month,
 SUM (CASE WHEN CAST (DISTANCE AS REAL) <= 0 THEN 1 ELSE 0 END) AS invalid_dist, 
 SUM (CASE WHEN CAST (CRS_DEP_TIME AS INTEGER) NOT BETWEEN 0 AND 2400 THEN 1 ELSE 0 END) AS  invalid_dep_range,
 SUM (CASE WHEN CAST (CRS_DEP_TIME AS INTEGER) % 100 NOT BETWEEN 0 AND 59 THEN 1 ELSE 0 END) AS invalid_dep_min, 
 SUM (CASE WHEN ARR_DEL15 NOT IN (0,1) THEN 1 ELSE 0 END) AS invalid_arr_del 
FROM flights_raw
WHERE 
CANCELLED=0
AND DIVERTED=0;
"""
pd.read_sql_query (q9_invalids, conn) 

,invalid_month,invalid_week,invalid_month,invalid_dist,invalid_dep_range,invalid_dep_min,invalid_arr_del
0,0,0,0,0,0,0,0


In [13]:
q10_duplicates= """
SELECT 
 FL_DATE, 
 TAIL_NUM,
 OP_UNIQUE_CARRIER,
 ORIGIN,
 DEST, 
 CRS_DEP_TIME,
 COUNT (*) AS duplicate_count
FROM flights_raw 
WHERE CANCELLED=0 AND DIVERTED=0
GROUP BY
 FL_DATE, 
 TAIL_NUM,
 OP_UNIQUE_CARRIER,
 ORIGIN,
 DEST, 
 CRS_DEP_TIME
HAVING COUNT(*) >1
ORDER BY duplicate_count DESC LIMIT 20;"""

pd.read_sql_query(q10_duplicates, conn)

,FL_DATE,TAIL_NUM,OP_UNIQUE_CARRIER,ORIGIN,DEST,CRS_DEP_TIME,duplicate_count


In [14]:
 #preview all rows to keep 
 q11_preview_rows="""
 SELECT
 MONTH,
 DAY_OF_MONTH,
 DAY_OF_WEEK,
 OP_UNIQUE_CARRIER,
 ORIGIN,
 DEST,
 CRS_DEP_TIME,
 DISTANCE,
 ARR_DEL15,
 CANCELLED,
 DIVERTED
 FROM flights_raw 
 WHERE DIVERTED=0
 AND CANCELLED=0
 AND ARR_DEL15 IS NOT NULL
 LIMIT 20;"""

 pd.read_sql_query (q11_preview_rows, conn) 

,MONTH,DAY_OF_MONTH,DAY_OF_WEEK,OP_UNIQUE_CARRIER,ORIGIN,DEST,CRS_DEP_TIME,DISTANCE,ARR_DEL15,CANCELLED,DIVERTED
0,1,1,3,AA,JFK,SFO,600,2586.0,0.0,0.0,0.0
1,1,1,3,AA,SFO,JFK,1030,2586.0,0.0,0.0,0.0
2,1,1,3,AA,SAT,CLT,819,1095.0,0.0,0.0,0.0
3,1,1,3,AA,BOS,LAX,801,2611.0,0.0,0.0,0.0
4,1,1,3,AA,JFK,LAX,2100,2475.0,0.0,0.0,0.0
5,1,1,3,AA,LAX,JFK,1130,2475.0,0.0,0.0,0.0
6,1,1,3,AA,CMH,PHX,746,1670.0,0.0,0.0,0.0
7,1,1,3,AA,PHX,OMA,2025,1037.0,0.0,0.0,0.0
8,1,1,3,AA,PHX,SMF,1104,647.0,1.0,0.0,0.0
9,1,1,3,AA,SMF,PHX,1259,647.0,0.0,0.0,0.0


- Initial data checks confirmed that the flight dates, month coverage, column values, and key attributes were within expected ranges. No missing values, invalid values, or duplicate flight records were found in the fields required for modeling.

### Create Cleaned Flight Table

This step creates `flights_cleaned` from the raw BTS flight table. The cleaned table keeps only the columns needed for modeling, removes cancelled and diverted flights, standardizes airport and airline codes, creates a clean flight date, and creates features needed for modelling such as route and departure hour.

Rows with missing or invalid key fields are removed so the final flight table stays flight level and ready to merge with weather data.

In [15]:
q12_create_flights_cleaned= """
DROP TABLE IF EXISTS flights_cleaned;

CREATE TABLE flights_cleaned AS 
SELECT 
 printf('2025-%02d-%02d', MONTH, DAY_OF_MONTH) AS FLIGHT_DATE, 
 MONTH AS MONTH, 
 DAY_OF_MONTH AS DAY_OF_MONTH, 
 DAY_OF_WEEK AS DAY_OF_WEEK, 
 UPPER(TRIM(ORIGIN)) AS ORIGIN, 
 UPPER(TRIM(DEST)) AS DESTINATION, 
 UPPER(TRIM(OP_UNIQUE_CARRIER)) AS AIRLINE, 
 UPPER(TRIM(ORIGIN))  || '-' || UPPER(TRIM(DEST)) AS ROUTE, 

 CAST(CRS_DEP_TIME AS INTEGER) AS SCHEDULED_DEP_TIME, 
 CASE 
  WHEN CAST(CRS_DEP_TIME AS INTEGER) = 2400 THEN 0 
  ELSE CAST(CAST(CRS_DEP_TIME AS INTEGER) / 100 AS INTEGER) 
 END AS DEPARTURE_HOUR, 
 CAST (DISTANCE AS REAL) AS DISTANCE,
 CAST (ARR_DEL15 AS INTEGER) AS IS_DELAYED
FROM flights_raw
WHERE 
CANCELLED=0 
 AND DIVERTED=0
 AND ARR_DEL15 IN (1,0)
 AND MONTH BETWEEN 1 AND 12
 AND DAY_OF_MONTH BETWEEN 1 AND 31
 AND DAY_OF_WEEK BETWEEN 1 AND 7
 AND OP_UNIQUE_CARRIER IS NOT NULL
 AND TRIM(OP_UNIQUE_CARRIER)<>''
 AND ORIGIN IS NOT NULL
 AND TRIM(ORIGIN)<>''
 AND DEST IS NOT NULL
 AND TRIM(DEST)<>''
 AND CRS_DEP_TIME IS NOT NULL 
 AND CAST (CRS_DEP_TIME AS INTEGER) BETWEEN 0 AND 2400
 AND CAST (CRS_DEP_TIME AS INTEGER) % 100 BETWEEN 0 AND 59
 AND DISTANCE IS NOT NULL
 AND CAST (DISTANCE AS REAL) >0;"""

conn.executescript(q12_create_flights_cleaned)
conn.commit() 

Final Data Health Check:

In [18]:
q13_check_flights_cleaned_= """
SELECT 
 COUNT(*) AS ROW_COUNT, 
 MIN(FLIGHT_DATE) AS START_DATE,
 MAX(FLIGHT_DATE) AS END_DATE,
 
    SUM (CASE WHEN FLIGHT_DATE IS NULL THEN 1 ELSE 0 END) AS FLIGHT_DATE_MISSING, 
    SUM (CASE WHEN MONTH IS NULL THEN 1 ELSE 0 END) AS MONTH_MISSING, 
    SUM (CASE WHEN DAY_OF_MONTH IS NULL THEN 1 ELSE 0 END) AS DAYOFMONTH_MISSING,
    SUM (CASE WHEN DAY_OF_WEEK IS NULL THEN 1 ELSE 0 END) AS DAYOFWEEK_MISSING, 
    SUM (CASE WHEN SCHEDULED_DEP_TIME IS NULL THEN 1 ELSE 0 END) AS DEPTIME_MISISNG,
    SUM (CASE WHEN DESTINATION IS NULL THEN 1 ELSE 0 END) AS DESTINATION_MISSING, 
    SUM (CASE WHEN DISTANCE IS NULL THEN 1 ELSE 0 END) AS DISTANCE_MISSING, 
    SUM (CASE WHEN ORIGIN IS NULL THEN 1 ELSE 0 END) AS ORIGIN_MISSING,
    SUM (CASE WHEN IS_DELAYED IS NULL THEN 1 ELSE 0 END) AS DELAYED_MISSING, 
    SUM (CASE WHEN ROUTE IS NULL THEN 1 ELSE 0 END) AS ROUTE_MISSING,
    SUM (CASE WHEN DEPARTURE_HOUR IS NULL THEN 1 ELSE 0 END) AS DEPHOUR_MISSING, 
    SUM (CASE WHEN AIRLINE IS NULL THEN 1 ELSE 0 END) AS AIRLINE_MISSING
FROM flights_cleaned;"""

pd.read_sql_query(q13_check_flights_cleaned_, conn)

,ROW_COUNT,START_DATE,END_DATE,FLIGHT_DATE_MISSING,MONTH_MISSING,DAYOFMONTH_MISSING,DAYOFWEEK_MISSING,DEPTIME_MISISNG,DESTINATION_MISSING,DISTANCE_MISSING,ORIGIN_MISSING,DELAYED_MISSING,ROUTE_MISSING,DEPHOUR_MISSING,AIRLINE_MISSING
0,2188776,2025-01-01,2025-04-30,0,0,0,0,0,0,0,0,0,0,0,0


In [20]:
q14_check_delayed= """
SELECT IS_DELAYED, COUNT(*) AS DELAYED_ROW_COUNT FROM flights_cleaned 
GROUP BY IS_DELAYED;
"""
pd.read_sql_query(q14_check_delayed, conn)

,IS_DELAYED,DELAYED_ROW_COUNT
0,0,1757906
1,1,430870


In [22]:
q15_final_preview= """
SELECT * FROM flights_cleaned LIMIT 10;"""

pd.read_sql_query(q15_final_preview, conn)

,FLIGHT_DATE,MONTH,DAY_OF_MONTH,DAY_OF_WEEK,ORIGIN,DESTINATION,AIRLINE,ROUTE,SCHEDULED_DEP_TIME,DEPARTURE_HOUR,DISTANCE,IS_DELAYED
0,2025-01-01,1,1,3,JFK,SFO,AA,JFK-SFO,600,6,2586.0,0
1,2025-01-01,1,1,3,SFO,JFK,AA,SFO-JFK,1030,10,2586.0,0
2,2025-01-01,1,1,3,SAT,CLT,AA,SAT-CLT,819,8,1095.0,0
3,2025-01-01,1,1,3,BOS,LAX,AA,BOS-LAX,801,8,2611.0,0
4,2025-01-01,1,1,3,JFK,LAX,AA,JFK-LAX,2100,21,2475.0,0
5,2025-01-01,1,1,3,LAX,JFK,AA,LAX-JFK,1130,11,2475.0,0
6,2025-01-01,1,1,3,CMH,PHX,AA,CMH-PHX,746,7,1670.0,0
7,2025-01-01,1,1,3,PHX,OMA,AA,PHX-OMA,2025,20,1037.0,0
8,2025-01-01,1,1,3,PHX,SMF,AA,PHX-SMF,1104,11,647.0,1
9,2025-01-01,1,1,3,SMF,PHX,AA,SMF-PHX,1259,12,647.0,0


- Final `flights_cleaned` table recorded with 2188776 cleaned rows (intial row count: 2229453), out of which 430870
include delayed flights; with no missing values & within the preferred date ranges. 